# Julián Serrano Chacón
# Mika Rodríguez Castro
  
# Práctica 3

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sounddevice as sd
import time # para medir tiempos de ejecución
import soundfile as sf     
from ipywidgets import interact
from ipywidgets import fixed
import sys
import scipy.signal as sg
from tkinter import *
from consts import *
from tkinter import *
from oscFM import *
from adsr import *
from synthFM import *

# graficos en el notebook
%matplotlib inline

In [2]:
root=Tk()
def key_pressed(event):
    key = event.char
    if key in teclas:
        index = teclas.index(key) # sacamos posición en el string notas
        nota = notas[index]
        pitch = pitchs[index]
        print(f'tecla {key} nota {nota} pitch {pitch}')
    elif key == '-':
        print('note off')

    root.bind("<Key>",key_pressed)
    root.mainloop()

In [3]:
%%writefile consts.py

# mapeo de teclas del ordenador a notas en el piano
# utilizamos '.' para los sostenidos
teclas = "zsxdcvgbhnjmq2w3er5t6y7u"  # 2 de teclas filas 
notas =  "C.D.EF.G.A.Bc.d.ef.g.a.b"  # mapeadas a 2 octavas
#         octava baja||octava alta


# frecuencias de las notas asociadas a las teclas del teclado
# partimos del la=220Hz y generamos frecuencias de escala temperada
pitchs = [ 220*2.0**(i/12.0) for i in range(len(teclas))] 

# frecuencias asociadas a las notas midi de 0 a 127
# El LA central es la nota midi 70 y su frecuencia es 440
# construimos hacia abajo y hacia arriba el resto de notas
freqsMidi = [ 440*2.0**(i/12.0) for i in range(-69,59)]


SRATE = 48000 # Sample rate, para todo el notebook

CHUNK = 1024


Overwriting consts.py


#### Ejercicio 1 (Obligatorio)  
Utilizar el controlador de teclado propuesto para implementar un pequeño instrumento monofónico que utilize el sintetizador FM visto en clase como generador de señal.  


In [16]:
synth = SynthFM()
input = None

def callback(outdata, frames, time, status):
    if status: print(status)
    # si hay generador de señal conectado, pedimos el siguiente bloque
    if input:    
        s = input.next()
        s = np.float32(s)
    # si no, generamos silencio    
    else:
        s = np.zeros(CHUNK,dtype=np.float32)

    outdata[:] = s.reshape(-1, 1)

# stream de salida con callBack
stream = sd.OutputStream(samplerate=SRATE, callback=callback, blocksize=CHUNK)
stream.start()

root=Tk()

# Caja de texto
text = Text(root,height=6,width=60)
text.pack(side=BOTTOM)
text.insert(INSERT,"Press keys\n")

def key_pressed(event):
    global synth
    global input
    key = event.char
    if key in teclas:
        index = teclas.index(key) # sacamos posición en el string notas
        synth = SynthFM(pitchs[index],beta=0.5)
        input = synth
    elif key == '-':
        synth.noteOff()

text.bind("<Key>",key_pressed)
root.mainloop()

# limpieza..
stream.stop()
stream.close()